# 07. Leave-one-day-out activity budget

**Verdict: keep the box model. Do not retry the GRU. Do not replace the boxes with the frozen-embedding fusion.**

Scripts: `../scripts/box_features.py`, `../scripts/activity_budget.py`, `../scripts/sequence_model.py`, `../scripts/frozen_fusion.py`. Run `python activity_budget.py --data <annotated-clips>` for the box budget. Add `--sequence` and `--fusion` to write all three models into one folder. Tables already saved: `../results/farm-protocol-suite/`.

Each model trains on five of the six days and is scored on the day left out. 2019-12-02 is not in the training set and not in the mean. Rows are the 2 Hz annotation grid. A row is kept when it is a stored label, or an interpolated box whose neighbouring human boxes are at most 20 raw frames apart, overlap by at least half, and share a label, or a trail of at most 10 raw frames. That is 15,177 kept rows and 9,486 stored rows.

The published dense-table scores of 0.321 and 0.360 use all seven days. This table uses six days and the 2 Hz annotation grid, so the two are not ranked against each other.

The decision number is the pen budget error, in percentage points. Lower is better. Copying the other days' budget, with no model, scores 12.24.

| model | stored macro-F1 | pen budget error | clip-pig budget error | verdict |
| --- | ---: | ---: | ---: | --- |
| Nine box features | 0.281 ± 0.085 | 10.40 | 12.78 | keep |
| Eight-step GRU on those features | 0.174 ± 0.036 | 14.13 | 18.03 | do not retry |
| Frozen embeddings plus the nine features | 0.284 ± 0.042 | 11.21 | 13.51 | do not retry |
| No model |  | 12.24 |  | baseline |

The one-date score of 0.4038 is the easy fold. On this protocol, 2019-11-11 is 0.358 for the box model and 2019-12-10 is 0.153.

The GRU reads the previous eight annotation steps, about four seconds, for five epochs. It loses to the copied budget. The fusion keeps MobileNet frozen, joins its 576 numbers to the nine box numbers, and fits logistic regression. Sleep recall rises from 0.285 to 0.514. The pen budget gets worse, 11.21 against 10.40. Drink recall falls from 0.466 to 0.227.


In [ ]:
from pathlib import Path
import pandas as pd

def results(name):
    here = Path.cwd()
    candidates = [
        here / "projects" / "pig-video" / "results" / name,
        here.parent / "results" / name,
        here / "results" / name,
    ]
    for path in candidates:
        if path.is_dir():
            return path
    raise FileNotFoundError(name)
folder = results('farm-protocol-suite')
display(pd.read_csv(folder / 'history.csv'))
display(pd.read_csv(folder / 'lodo_folds.csv'))
report = pd.read_csv(folder / 'validation_report.csv', index_col=0)
display(report)
